In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI
import requests

In [2]:
# models
model_gemini='gemini-3-flash-preview'
model_gpt = 'gpt-4o-mini'
model_llama = 'llama3.2'


In [3]:
#creating gemini client

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini")
elif not google_api_key.startswith("AIz"):
    print("An API key was found, but it doesn't start AIz")
else:
    print("API key found and looks good so far!")


gemini=OpenAI(api_key=google_api_key,base_url=GEMINI_BASE_URL)
print("your gemini client is ready to use")

API key found and looks good so far!
your gemini client is ready to use


In [4]:
requests.get("http://localhost:11434").content

b'Ollama is running'

In [5]:
#creating Ollama client

OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
print("your ollama client is ready to use")

your ollama client is ready to use


In [6]:
relevant_link_system_prompt="""   
You are a marketing assistant specialized in selecting relevant links from a list of URLs.

Your task:
- Analyze the provided URLs.
- Select only the links that are useful for a marketing brochure.

Ignore:
- Login / Signup
- Privacy Policy
- Terms & Conditions
- Admin / Dashboard
- Duplicate or irrelevant links

Output Rules:
- Return ONLY valid JSON (no explanations, no extra text).
- Use EXACTLY this format:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}

Type Rules:
- Use clear lowercase values such as:
  "about page", "product page", "pricing page", "contact page",
  "blog", "case studies", "testimonials", "careers page"


- No duplicates

Additional instruction:
- Infer the page type from the URL if not explicitly clear.

"""

In [7]:

def get_relevant_links_user_prompt(company_links):
     relevant_links_user_prompt=f""""Company Name: 

Here is a list of URLs from the company website:

{company_links}

Select the most relevant links for a marketing brochure based on their usefulness to customers.

Return ONLY JSON as specified.

"""
     return relevant_links_user_prompt


In [26]:

def get_relevant_links(url):
    #fetching website links
    company_links=fetch_website_links(url)
   
    #sending request
    response = gemini.chat.completions.create(model=model_gemini, messages=[{"role": "system", "content":relevant_link_system_prompt},{"role":"user","content": get_relevant_links_user_prompt(company_links)}],response_format={"type":"json_object"})
    
    #loading links in json object
    links = json.loads(response.choices[0].message.content)
    return links
   

In [9]:
get_relevant_links(url="https://www.udemy.com/")

{'links': [{'type': 'about page', 'url': 'https://about.udemy.com/stories/'},
  {'type': 'product page',
   'url': 'https://www.udemy.com/course/ios-13-app-development-bootcamp/'},
  {'type': 'product page',
   'url': 'https://www.udemy.com/course/aws-certified-developer-associate-dva-c01/'},
  {'type': 'learning path',
   'url': 'https://www.udemy.com/partner/google/ai-professional-certificate-learning-path/'}]}

In [25]:
from sys import exception


def fetch_relevant_links_content(url):
    result=''
    try :

        contents = fetch_website_contents(url)
        relevant_links = get_relevant_links(url)
        result = f"## Landing Page:\n\n{contents}\n##Relevant  Links:\n"
        for link in relevant_links['links']:
            try :
                 result += f"\n\n### Link: {link['type']}\n"
                 result += fetch_website_contents(link["url"])
            except Exception as e:{}
    except Exception as e:{}

    return result

In [11]:
print(fetch_relevant_links_content("https://www.udemy.com/"))

## Landing Page:

Udemy: Online Courses for Skills, Careers & AI

Search bar
Search for anything
Site navigation
Learn AI with Google
Find Courses
Learn AI
Get Certified
Courses with Role Play
More from Udemy
Udemy Business
Get the app
Invite friends
Help and Support
English
Menu
About Udemy Business
Compare Plans
Try Udemy Business
Skip to content
Categories
Search for anything
Learn
essential
career and
life
skills
Udemy helps you build in-demand skills fast and advance your career in a changing job market
Show more
Show less
Trusted by over 17,000 companies and millions of learners around the world
Join others transforming their lives through learning
The course did a great job explaining AI - from development through application. I appreciated the varying perspectives presented, which were helpful in understanding how to use AI responsibly as a tool in my profession, rather than a novelty.
Cris M.
Google AI Essentials graduate
View AI courses
Udemy was truly
a game-changer and a gr

In [28]:
brochure_system_prompt = """You are an expert marketing copywriter and brand strategist.

Your task is to generate a high-quality, visually appealing, and persuasive marketing brochure based on provided website content.

Guidelines:
- Write in a professional, engaging, and concise tone.
- Focus on value proposition, benefits, and differentiation.
- Avoid copying text verbatim; rewrite and enhance it.
- Structure the brochure clearly with well-defined sections.
- Use simple, compelling language suitable for customers (avoid unnecessary technical jargon).
- Highlight key services, products, and strengths.
- Remove irrelevant, repetitive, or noisy content.
- If information is missing, infer reasonably but do not hallucinate specific facts.

✨ Visual & Style Requirements:
- Use tasteful special symbols and icons (e.g., 🚀 ✨ 📡 💡 🔹 ✅) to enhance readability and attractiveness.
- Keep formatting clean and professional (do NOT overuse symbols).
- Use bullet points where appropriate for clarity.
- Make headings stand out and easy to scan.

Output Format (Markdown, no code blocks):

# 🚀 Company Name

### ✨ Tagline
(A short, catchy phrase)

### 💡 About Us
(A compelling overview of the company)

### 📦 Products / Services
- 🔹 Service/Product 1
- 🔹 Service/Product 2
- 🔹 Service/Product 3

### ⚡ Key Features / Advantages
- ✅ Benefit 1
- ✅ Benefit 2
- ✅ Benefit 3

### 🏆 Why Choose Us
- 🌟 Unique strength 1
- 🌟 Unique strength 2
- 🌟 Unique strength 3

### 📞 Call to Action
(A strong, persuasive closing encouraging action)

Important:
- Do NOT include raw URLs or navigation elements unless useful.
- Do NOT mention “based on the provided content”.
- Ensure the brochure is polished, modern, and ready for real-world marketing use.
- Balance professionalism with visual appeal — avoid clutter.
"""

In [29]:
def get_brochure_user_prompt(url):
    website_content=fetch_relevant_links_content(url)
    user_prompt=f"""You are given content extracted from a company's website.



Website Content:
{website_content}

Instructions:
- Analyze the content and identify the company’s core business, services, and value proposition.
- Create a compelling marketing brochure using the provided information.
- Remove irrelevant elements such as menus, scripts, or duplicated text.
- Focus on clarity, persuasion, and customer appeal.
-make it attractiv with emotions symbols

Generate the brochure in markdown format."""

    return user_prompt
    

In [30]:
def stream_brochure(url):
    stream = gemini.chat.completions.create(model=model_gemini, messages=[{"role": "system", "content":brochure_system_prompt},{"role":"user","content": get_brochure_user_prompt(url)}],stream=True)

    response=""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
           response += chunk.choices[0].delta.content or ''
           update_display(Markdown(response), display_id=display_handle.display_id) 

In [31]:
stream_brochure("https://www.udemy.com/")

# 🚀 Udemy

### ✨ Master Your Future: In-Demand Skills for a Changing World
*Empowering millions to transform their lives through flexible, expert-led online learning.*

### 💡 About Us
Udemy is the leading global marketplace for learning and instruction. We bridge the gap between where you are and where you want to be by providing access to high-quality, real-world skills. Whether you are looking to master Artificial Intelligence, launch a new career in tech, or lead a global team, Udemy provides the tools and expertise to help you stay ahead in an ever-evolving job market.

### 📦 Key Learning Paths
- 🔹 **Artificial Intelligence (AI) with Google:** Build fluency with foundational skills, prompting techniques, and automation tools designed by Google experts.
- 🔹 **Tech & Development:** Master mobile app creation with the **iOS & Swift Bootcamp** or dive into cloud computing with **AWS Certification** paths.
- 🔹 **Professional Certifications:** Prepare for industry-standard exams from CompTIA, PMI, and AWS with comprehensive practice tests and exam vouchers.
- 🔹 **Udemy Business:** Tailored solutions for teams and organizations to stay competitive and upskill at scale.

### ⚡ The Competitive Advantage
- ✅ **Career-Ready Skills:** Gain practical experience through hands-on activities and real-world projects that you can apply immediately at work.
- ✅ **Industry-Recognized Credentials:** Earn certificates from world-class partners like **Google** and **AWS** to prove your expertise to employers.
- ✅ **Flexible Learning:** Study at your own pace, on any device, with lifetime access to thousands of lectures and resources.
- ✅ **Expert Instruction:** Learn directly from industry leaders, startup founders, and veteran developers who bring real-world experience to your screen.

### 🏆 Why Choose Udemy?
- 🌟 **Trusted by Global Leaders:** More than **17,000 companies** and millions of learners worldwide rely on Udemy for their professional development.
- 🌟 **Proven Impact:** Our AI-skilled learners report a significant competitive edge, with AI-related job postings increasing by **108%** over the past two years.
- 🌟 **Measurable Growth:** From $40K to $170K salary jumps to launching functional law-firm apps within 24 hours—our learners’ transformation stories speak for themselves.
- 🌟 **Massive Library:** Access over 60 hours of content in specialized bootcamps, covering everything from basic coding to advanced machine learning and augmented reality.

### 📞 Your New Chapter Starts Here
Stop waiting for the future—build it. Join the millions of professionals who are rewriting their stories through learning. Whether you want to master **Generative AI** or become a **Certified Cloud Developer**, the path to your next promotion starts today.

**🚀 Start Learning Now and Own Your Tomorrow.**